In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
import pandas as pd
pd.set_option('display.max_columns', None)
import numpy as np
import sys
import os
sys.path.append(os.path.join(os.getcwd(), '..', '..'))

from llm_good_enough import LLMGoodEnough


# PolitiFact S6 Analysis
**6-point scale dataset**

Data preprocessing is done in `preprocessing.ipynb`.


In [ ]:
# Load cleaned S6 data
merged_s6_df = pd.read_csv("../data/merged_s6_cleaned.csv")
print(f"Loaded S6 data: {merged_s6_df.shape}")

# Identify human columns
HUMAN_COLS = [c for c in merged_s6_df.columns if c.startswith("human_#")]
LLM_COLS = ["GPT-4o mini"]

# Determine score boundaries from human ratings
unique_human_ratings = np.unique(merged_s6_df[HUMAN_COLS].values[~pd.isna(merged_s6_df[HUMAN_COLS].values)])
MIN_SCORE, MAX_SCORE = int(np.min(unique_human_ratings)), int(np.max(unique_human_ratings))
print(f"Human rating scale: {unique_human_ratings}")

merged_s6_df.head()


## LLM vs Human Disagreement Barplot


In [ ]:
judge = LLMGoodEnough(
    df=merged_s6_df,
    human_cols=HUMAN_COLS,
    llm_cols=LLM_COLS,
    min_score=MIN_SCORE,
    max_score=MAX_SCORE,
    verbosity=1
)

judge.plot_judges_grid(
    save_path="../figures/svg/politifact_s6_barplot.svg"
)


## Monte Carlo Robustness Analysis


In [ ]:
judge = LLMGoodEnough(
    df=merged_s6_df,
    human_cols=HUMAN_COLS,
    llm_cols=LLM_COLS,
    min_score=MIN_SCORE, 
    max_score=MAX_SCORE,
)

judge.plot_monte_carlo_robustness(
    llm_col="GPT-4o mini",
    save_path="../figures/svg/politifact_s6_monte_carlo_robustness.svg"
)


In [ ]:
# P-value distribution analysis
counts = judge.count_pvalue_samples(iterations=25_000, threshold=0.05)

print("=" * 60)
print("P-value Distribution Analysis (S6)")
print("=" * 60)
print(f"Total Monte Carlo samples: {counts['total']:,}")
print(f"\nSamples with p-value > 0.05:  {counts['above']:,} ({counts['above_pct']:.2f}%)")
print(f"Samples with p-value ≤ 0.05:  {counts['below']:,} ({counts['below_pct']:.2f}%)")
print("=" * 60)


## Human Stability Analysis


In [ ]:
judge.plot_human_stability_analysis(
    convergence_threshold=0.000005,
    min_iterations=20_000,
    max_iterations=500_000,
    save_path="../figures/svg/politifact_s6_human_stability.svg",
    parallel=True,
)


In [ ]:
judge.plot_human_stability_analysis(
    percentages=[50, 100, 150, 200, 250, 300, 350, 400, 450, 500],
    convergence_threshold=0.000005,
    min_iterations=20_000,
    max_iterations=500_000,
    save_path="../figures/svg/politifact_s6_human_stability_100plus.svg",
    parallel=True,
)
